# Parte Comum: K-Means, Passada 2 (k fixo)

Dataset: Obesity (UCI 544), arquivo `data/parte_comum/obesity.csv`.

## Etapa 1: Padronização das variáveis

In [ ]:
# [IA] Claude Code – importacao das bibliotecas
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [ ]:
# [IA] Claude Code – leitura do dataset (caminho relativo a pasta do notebook)
CAMINHO_DADOS = Path("../../data/parte_comum/obesity.csv")

df = pd.read_csv(CAMINHO_DADOS)
print(df.shape)
df.head()

In [ ]:
# [IA] Claude Code – separar variaveis de entrada (X) e classe real (y)
# A classe real NAO entra no K-Means; fica guardada para o mapeamento e as metricas.
COLUNA_CLASSE = "NObeyesdad"

y = df[COLUNA_CLASSE].copy()
X = df.drop(columns=[COLUNA_CLASSE]).copy()

print("Classes reais:", y.nunique())
print(y.value_counts())

In [ ]:
# [IA] Claude Code – variaveis numericas usadas como entrada do K-Means (lista explicita)
# O K-Means agrupa por distancia euclidiana, que so faz sentido para variaveis numericas.
# As categoricas (Gender, family_history_with_overweight, FAVC, CAEC, SMOKE, SCC, CALC, MTRANS)
# continuam sendo analisadas na EDA; so nao entram no agrupamento.
# A Passada 1 (kmeans_passada1.ipynb) deve usar exatamente esta mesma lista.
VARIAVEIS_KMEANS = ["Age", "Height", "Weight", "FCVC", "NCP", "CH2O", "FAF", "TUE"]

X = X[VARIAVEIS_KMEANS]

# Conferencia: todas as variaveis escolhidas sao numericas
assert all(pd.api.types.is_numeric_dtype(X[c]) for c in VARIAVEIS_KMEANS), "Alguma variavel nao e numerica"
X.head()

In [ ]:
# [IA] Claude Code – padronizacao (z-score): media 0 e desvio padrao 1 em cada variavel
scaler = StandardScaler()
X_padronizado = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index,
)
X_padronizado.head()

In [ ]:
# [IA] Claude Code – conferir a padronizacao (media ~ 0 e desvio ~ 1)
conferencia = pd.DataFrame({
    "media_antes": X.mean(),
    "desvio_antes": X.std(ddof=0),
    "media_depois": X_padronizado.mean().round(6),
    "desvio_depois": X_padronizado.std(ddof=0).round(6),
})
conferencia

### Por que padronizar antes do K-Means?

_(texto do grupo)_

## Etapa 2: K-Means com k = número de classes reais

In [ ]:
# [IA] Claude Code – k fixo no numero de classes reais do dataset
from sklearn.cluster import KMeans

K_FIXO = y.nunique()
print("k fixo =", K_FIXO)

In [ ]:
# [IA] Claude Code – executar o K-Means sobre os dados padronizados
# random_state fixo para o resultado ser reproduzivel; n_init=10 roda 10 inicializacoes e fica com a melhor
kmeans_fixo = KMeans(n_clusters=K_FIXO, n_init=10, random_state=42)
clusters_fixo = kmeans_fixo.fit_predict(X_padronizado)

print("Inercia (soma das distancias ao quadrado):", round(kmeans_fixo.inertia_, 2))

In [ ]:
# [IA] Claude Code – guardar o cluster de cada linha junto com a classe real (usado no mapeamento e nas metricas)
resultado_fixo = pd.DataFrame({
    "cluster": clusters_fixo,
    "classe_real": y.values,
}, index=X.index)

# Quantidade de pontos em cada cluster
resultado_fixo["cluster"].value_counts().sort_index().rename("qtd_pontos").to_frame()

In [ ]:
# [IA] Claude Code – centroides de cada cluster na escala original das variaveis
centroides_fixo = pd.DataFrame(
    scaler.inverse_transform(kmeans_fixo.cluster_centers_),
    columns=X.columns,
).round(2)
centroides_fixo.index.name = "cluster"
centroides_fixo

## Etapa 3: Visualização dos grupos

In [ ]:
# [IA] Claude Code – reduzir as 8 variaveis padronizadas para 2 componentes (PCA) so para desenhar
# O K-Means continua rodando nas 8 variaveis; o PCA serve apenas para projetar os pontos no plano.
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_padronizado)
centroides_2d = pca.transform(kmeans_fixo.cluster_centers_)

var_explicada = pca.explained_variance_ratio_ * 100
print(f"Variancia explicada: PC1 = {var_explicada[0]:.1f}%, PC2 = {var_explicada[1]:.1f}%, "
      f"total = {var_explicada.sum():.1f}%")

In [ ]:
# [IA] Claude Code – grafico lado a lado: clusters do K-Means x classes reais, na mesma projecao PCA
ordem_classes = [
    "Insufficient_Weight", "Normal_Weight",
    "Overweight_Level_I", "Overweight_Level_II",
    "Obesity_Type_I", "Obesity_Type_II", "Obesity_Type_III",
]
cores = plt.get_cmap("tab10").colors

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), sharex=True, sharey=True)

for i, cluster in enumerate(sorted(resultado_fixo["cluster"].unique())):
    filtro = resultado_fixo["cluster"].values == cluster
    ax1.scatter(coords[filtro, 0], coords[filtro, 1], s=12, alpha=0.6,
                color=cores[i], label=f"Cluster {cluster}")
ax1.scatter(centroides_2d[:, 0], centroides_2d[:, 1], marker="X", s=200,
            color="black", edgecolor="white", linewidth=1.5, label="Centroide")
ax1.set_title(f"Passada 2: clusters do K-Means (k = {K_FIXO})")

for i, classe in enumerate(ordem_classes):
    filtro = y.values == classe
    ax2.scatter(coords[filtro, 0], coords[filtro, 1], s=12, alpha=0.6,
                color=cores[i], label=classe)
ax2.set_title("Classes reais (NObeyesdad)")

for ax in (ax1, ax2):
    ax.set_xlabel(f"Componente principal 1 ({var_explicada[0]:.1f}% da variancia)")
    ax.legend(fontsize=8, markerscale=1.5, loc="best")
ax1.set_ylabel(f"Componente principal 2 ({var_explicada[1]:.1f}% da variancia)")

fig.suptitle("Obesity: grupos encontrados x classes reais (projecao PCA das 8 variaveis padronizadas)")
fig.tight_layout()

PASTA_SAIDA = Path("../../out/parte_comum")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
fig.savefig(PASTA_SAIDA / "passada2_clusters_pca.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# [IA] Claude Code – quanto cada variavel pesa em cada componente (ajuda a ler os eixos do grafico)
pd.DataFrame(
    pca.components_.T,
    index=VARIAVEIS_KMEANS,
    columns=["PC1", "PC2"],
).round(3)